<a href="https://colab.research.google.com/github/Erick88santos/Dataset_shark_anotation/blob/main/DRONESHARK_4FOLD_v2_EXPERIMENTOS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DroneShark — 4-Fold agrupado, versão 2 (experimentos de ajuste)

Esta versão serve para testar **ajustes baratos** sem recomeçar do zero e **sem contaminar o teste**:

- **Grade de limiares até 0,001** (antes parava em 0,05).
- **Experimentos de configuração** (`EXP`): `yolo11s_640` (já feito), `yolo11s_960` (resolução maior) e `yolo11m_640` (modelo maior).
- **Os mesmos folds** em todos os experimentos (lidos de `folds_atribuicao.csv`), para a comparação ser justa.
- **Cache de pontuações**: depois que um fold foi avaliado, mudar a grade de limiares custa segundos, sem rodar o modelo de novo.
- **Regra anti-viés:** a configuração vencedora é escolhida pela **acurácia na validação interna**, nunca no teste. No TCC, todas as configurações testadas entram na tabela.

**Como usar:** ajuste `EXP` na célula de configuração, rode o notebook inteiro, e repita para cada experimento (reiniciando a sessão entre eles ou rodando tudo de novo). A última seção compara todos.

In [1]:
!pip -q install ultralytics scikit-learn

In [2]:
import os, json, random, shutil, zipfile, math
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
from scipy.fftpack import dct

# ===================== CONFIGURAÇÃO =====================
EXP = "yolo11m_640"        # <<< TROQUE AQUI: "yolo11s_640" | "yolo11s_960" | "yolo11m_640"

EXPERIMENTOS = {
    "yolo11s_640": dict(modelo="yolo11s.pt", imgsz=640, batch=16),   # configuração original
    "yolo11s_960": dict(modelo="yolo11s.pt", imgsz=960, batch=8),    # resolução maior (objetos pequenos)
    "yolo11m_640": dict(modelo="yolo11m.pt", imgsz=640, batch=16),   # modelo maior
}
CFG = EXPERIMENTOS[EXP]
MODELO, IMGSZ, BATCH = CFG["modelo"], CFG["imgsz"], CFG["batch"]

SEED = 42
N_FOLDS = 4
VAL_N_SPLITS = 6
EPOCHS, PATIENCE = 100, 20

PHASH_SIZE, PHASH_MAX_DIST = 16, 20

# Grade de limiares: agora desce até 0,001 (a anterior parava em 0,05)
CONF_CANDIDATOS = np.round(np.concatenate([[0.001, 0.005, 0.01, 0.02, 0.03, 0.04],
                                           np.arange(0.05, 0.951, 0.025)]), 4)
CONF_FIXOS = {"conf 0.50 (estudo anterior)": 0.50, "conf 0.25 (padrão YOLO)": 0.25}
M_ACC = "limiar escolhido na validação (acurácia)"
M_BAL = "limiar escolhido na validação (acur. balanceada)"

DRIVE = Path("/content/drive/MyDrive/droneshark")
ZIP_DATASET = DRIVE / "dataset_shark_binario.zip"
BASE = DRIVE / "resultados_4fold"          # pasta comum a todos os experimentos
SAIDA = BASE / EXP                          # pasta deste experimento
LOCAL_RUNS = Path("/content/runs") / EXP    # treino em disco local (o Drive corrompe checkpoints)
# ========================================================
random.seed(SEED); np.random.seed(SEED)
print("Experimento:", EXP, CFG)

Experimento: yolo11m_640 {'modelo': 'yolo11m.pt', 'imgsz': 640, 'batch': 16}


In [3]:
from google.colab import drive
drive.mount('/content/drive')
SAIDA.mkdir(parents=True, exist_ok=True)

DATA = Path("/content/dataset_shark_binario")
if not DATA.exists():
    zipfile.ZipFile(ZIP_DATASET).extractall("/content")

imgs = sorted(p for p in (DATA/"images").glob("*") if p.is_file())
linhas = []
for p in imgs:
    lab = DATA/"labels"/f"{p.stem}.txt"
    assert lab.exists(), f"Sem label para {p.name}"
    linhas.append({"img": str(p), "nome": p.name, "y": int(lab.read_text().strip() != "")})
df = pd.DataFrame(linhas)
n, pos = len(df), int(df.y.sum())
print(f"Total: {n} | com tubarão: {pos} | sem tubarão: {n-pos} | baseline: {pos/n:.1%}")

Mounted at /content/drive
Total: 875 | com tubarão: 731 | sem tubarão: 144 | baseline: 83.5%


## 1. Grupos e folds (os MESMOS em todos os experimentos)

In [4]:
from sklearn.model_selection import StratifiedGroupKFold
FOLDS_CSV = BASE / "folds_atribuicao.csv"

def phash_bits(caminho, hash_size=PHASH_SIZE, fator=4):
    tam = hash_size * fator
    with Image.open(caminho) as im:
        a = np.asarray(im.convert("L").resize((tam, tam), Image.LANCZOS), dtype=np.float32)
    d = dct(dct(a, axis=0), axis=1)
    baixa = d[:hash_size, :hash_size]
    return (baixa > np.median(baixa)).flatten().astype(np.float32)

if FOLDS_CSV.exists():
    ref = pd.read_csv(FOLDS_CSV)[["nome", "grupo", "fold"]]
    df = df.merge(ref, on="nome", how="left")
    assert df.fold.notna().all(), "folds_atribuicao.csv não cobre todas as imagens"
    df["fold"] = df.fold.astype(int)
    print("Folds carregados de folds_atribuicao.csv (iguais aos dos experimentos anteriores).")
else:
    H = np.stack([phash_bits(p) for p in df.img]); soma = H.sum(1)
    dist = soma[:, None] + soma[None, :] - 2 * (H @ H.T)
    pai = list(range(len(df)))
    def achar(x):
        while pai[x] != x:
            pai[x] = pai[pai[x]]; x = pai[x]
        return x
    for i, j in zip(*np.where(np.triu(dist <= PHASH_MAX_DIST, 1))):
        ri, rj = achar(i), achar(j)
        if ri != rj: pai[rj] = ri
    df["grupo"] = [achar(i) for i in range(len(df))]
    sg = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    df["fold"] = -1
    for k, (_, te) in enumerate(sg.split(df, df.y, df.grupo)):
        df.loc[df.index[te], "fold"] = k
    df.to_csv(FOLDS_CSV, index=False)
    print("Folds criados e salvos em folds_atribuicao.csv.")

assert (df.groupby("grupo").fold.nunique() == 1).all(), "Há grupos divididos entre folds!"
resumo = df.groupby("fold").agg(imagens=("y", "size"), com_tubarao=("y", "sum"), grupos=("grupo", "nunique"))
resumo["sem_tubarao"] = resumo.imagens - resumo.com_tubarao
resumo["baseline_sempre_tubarao"] = (resumo.com_tubarao / resumo.imagens).map("{:.1%}".format)
display(resumo)

Folds carregados de folds_atribuicao.csv (iguais aos dos experimentos anteriores).


,imagens,com_tubarao,grupos,sem_tubarao,baseline_sempre_tubarao
fold,,,,,
0,214,178,141,36,83.2%
1,227,196,142,31,86.3%
2,218,180,143,38,82.6%
3,216,177,144,39,81.9%


## 2. Funções

In [5]:
import yaml
from ultralytics import YOLO

def montar_fold(k):
    base = Path(f"/content/folds/fold{k}"); base.mkdir(parents=True, exist_ok=True)
    teste = df[df.fold == k]
    resto = df[df.fold != k].reset_index(drop=True)
    sg = StratifiedGroupKFold(n_splits=VAL_N_SPLITS, shuffle=True, random_state=SEED + k)
    tr, va = next(sg.split(resto, resto.y, resto.grupo))
    treino, val = resto.iloc[tr], resto.iloc[va]
    gt, gv, gs = set(treino.grupo), set(val.grupo), set(teste.grupo)
    assert not (gt & gv) and not (gt & gs) and not (gv & gs), "Vazamento de grupos!"
    for nome, parte in [("train", treino), ("val", val), ("test", teste)]:
        (base/f"{nome}.txt").write_text("\n".join(parte.img))
    yml = base/"data.yaml"
    yml.write_text(yaml.safe_dump({"train": str(base/"train.txt"), "val": str(base/"val.txt"),
                                   "test": str(base/"test.txt"), "names": {0: "shark"}}))
    return {"yaml": yml, "treino": treino, "val": val, "teste": teste}

def pesos_validos(caminho):
    try:
        import torch
        torch.load(str(caminho), map_location="cpu", weights_only=False)
        return True
    except Exception:
        return False

def treinar(k, info):
    drive_fold = SAIDA/f"fold{k}"
    best_drive, marca = drive_fold/"weights"/"best.pt", drive_fold/"treino_ok.txt"
    if marca.exists() and best_drive.exists() and pesos_validos(best_drive):
        print(f"Fold {k}: pesos já existem no Drive, reaproveitando."); return best_drive
    shutil.rmtree(drive_fold, ignore_errors=True)
    shutil.rmtree(LOCAL_RUNS/f"fold{k}", ignore_errors=True)
    YOLO(MODELO).train(data=str(info["yaml"]), epochs=EPOCHS, patience=PATIENCE, imgsz=IMGSZ,
                       batch=BATCH, seed=SEED, deterministic=True, project=str(LOCAL_RUNS),
                       name=f"fold{k}", exist_ok=True, plots=False)
    local = LOCAL_RUNS/f"fold{k}"
    (drive_fold/"weights").mkdir(parents=True, exist_ok=True)
    for nome in ["best.pt", "last.pt"]:
        shutil.copy2(local/"weights"/nome, drive_fold/"weights"/nome)
    if (local/"results.csv").exists():
        shutil.copy2(local/"results.csv", drive_fold/"results.csv")
    assert pesos_validos(drive_fold/"weights"/"best.pt"), "best.pt no Drive está inválido"
    marca.write_text("ok")
    return local/"weights"/"best.pt"

def scores_imagem(modelo, caminhos, lote=32):
    out = []
    for i in range(0, len(caminhos), lote):
        for r in modelo.predict(list(caminhos[i:i+lote]), conf=0.001, imgsz=IMGSZ, verbose=False):
            out.append(float(r.boxes.conf.max()) if len(r.boxes) else 0.0)
    return np.array(out)

def metricas(y, s, thr):
    y = np.asarray(y); pred = (np.asarray(s) >= thr).astype(int)
    tp = int(((pred == 1) & (y == 1)).sum()); fp = int(((pred == 1) & (y == 0)).sum())
    fn = int(((pred == 0) & (y == 1)).sum()); tn = int(((pred == 0) & (y == 0)).sum())
    div = lambda a, b: a / b if b else 0.0
    prec, rec, esp = div(tp, tp+fp), div(tp, tp+fn), div(tn, tn+fp)
    return {"limiar": float(thr), "TP": tp, "FP": fp, "FN": fn, "TN": tn,
            "acuracia": div(tp+tn, tp+fp+fn+tn), "precisao": prec, "recall": rec,
            "especificidade": esp, "acuracia_balanceada": (rec+esp)/2,
            "f1": div(2*prec*rec, prec+rec)}

def escolher_limiar(y, s, criterio, grade=None):
    # Escolhe na VALIDAÇÃO INTERNA; empate: menor limiar (favorece o Recall)
    grade = CONF_CANDIDATOS if grade is None else grade
    melhor = None
    for t in grade:
        v = metricas(y, s, t)[criterio]
        if melhor is None or v > melhor[0] + 1e-12:
            melhor = (v, t)
    return float(melhor[1])

def wilson(k, n, z=1.96):
    if n == 0: return (0.0, 0.0)
    p = k / n; d = 1 + z*z/n
    c = (p + z*z/(2*n)) / d
    h = z * math.sqrt(p*(1-p)/n + z*z/(4*n*n)) / d
    return (c - h, c + h)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings


## 3. Etapa 1 — treinar (se preciso) e guardar as pontuações

Para cada fold, o notebook salva em `scores_foldK.json` as pontuações (confiança máxima por imagem) da **validação interna** e do **teste**, mais as métricas Box-Level. Depois disso, trocar a grade de limiares não exige rodar o modelo de novo.

In [6]:
def etapa1_fold(k):
    cache = SAIDA/f"scores_fold{k}.json"
    if cache.exists():
        print(f"Fold {k}: pontuações em cache."); return json.loads(cache.read_text())
    print(f"\n================ FOLD {k} ({EXP}) ================")
    info = montar_fold(k)
    print(f"treino={len(info['treino'])} | validação interna={len(info['val'])} | teste={len(info['teste'])}")
    best = treinar(k, info)
    modelo = YOLO(str(best))
    s_val = scores_imagem(modelo, info["val"].img.tolist())
    s_te = scores_imagem(modelo, info["teste"].img.tolist())
    v = modelo.val(data=str(info["yaml"]), split="test", imgsz=IMGSZ, batch=BATCH, conf=0.001,
                   verbose=False, plots=False)
    out = {"fold": k,
           "val": {"nome": info["val"].nome.tolist(), "y": info["val"].y.tolist(), "score": s_val.tolist()},
           "test": {"nome": info["teste"].nome.tolist(), "y": info["teste"].y.tolist(), "score": s_te.tolist()},
           "box": {"fold": k, "precisao": float(v.box.mp), "recall": float(v.box.mr),
                   "mAP50": float(v.box.map50), "mAP50_95": float(v.box.map)}}
    cache.write_text(json.dumps(out))
    return out

dados = [etapa1_fold(k) for k in range(N_FOLDS)]
print("\nEtapa 1 concluída.")


================ FOLD 0 (yolo11m_640) ================
treino=554 | validação interna=107 | teste=214
Fold 0: pesos já existem no Drive, reaproveitando.
Ultralytics 8.4.207 🚀 Python-3.13.18 torch-2.14.1+cpu CPU (Intel Xeon CPU @ 2.20GHz)
YOLO11m summary (fused): 125 layers, 20,030,803 parameters, 0 gradients, 67.8 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 821.0±371.9 MB/s, size: 51.2 KB)
val: Scanning /content/dataset_shark_binario/labels... 214 images, 36 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 214/214 780.5it/s 0.3s
val: New cache created: /content/dataset_shark_binario/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 14/14 20.4s/it 4:46
                   all        214        359      0.573      0.476      0.482      0.197
Speed: 5.7ms preprocess, 1315.6ms inference, 0.0ms loss, 1.1ms postprocess per image

================ FOLD 1 (yolo11m_640) ================
treino=541 | validação interna=1

## 4. Etapa 2 — aplicar os limiares (rápido; pode ser repetida com outra grade)

In [7]:
def avaliar(dados, grade=None):
    linhas = []
    for d in dados:
        y_val, s_val = np.array(d["val"]["y"]), np.array(d["val"]["score"])
        y_te, s_te = np.array(d["test"]["y"]), np.array(d["test"]["score"])
        limiares = dict(CONF_FIXOS)
        limiares[M_ACC] = escolher_limiar(y_val, s_val, "acuracia", grade)
        limiares[M_BAL] = escolher_limiar(y_val, s_val, "acuracia_balanceada", grade)
        for metodo, thr in limiares.items():
            m = metricas(y_te, s_te, thr); mv = metricas(y_val, s_val, thr)
            m.update({"experimento": EXP, "fold": d["fold"], "metodo": metodo,
                      "baseline": float(y_te.mean()), "n_teste": int(len(y_te)),
                      "val_acuracia": mv["acuracia"], "val_acur_balanceada": mv["acuracia_balanceada"]})
            linhas.append(m)
    return pd.DataFrame(linhas)

res = avaliar(dados)
box = pd.DataFrame([d["box"] for d in dados])
res.to_csv(SAIDA/"resultados_por_fold.csv", index=False)
box.to_csv(SAIDA/"box_level_por_fold.csv", index=False)

cols = ["acuracia", "precisao", "recall", "especificidade", "acuracia_balanceada", "f1"]
linhas_tab = []
for metodo, g in res.groupby("metodo", sort=False):
    linha = {"Método de decisão": metodo}
    for c in cols: linha[c] = f"{g[c].mean():.2%} ± {g[c].std(ddof=1):.2%}"
    linha["baseline"] = f"{g.baseline.mean():.2%}"
    linha["acurácia − baseline"] = f"{(g.acuracia - g.baseline).mean()*100:+.2f} p.p."
    linha["limiares por fold"] = ", ".join(f"{x:.3f}" for x in g.limiar)
    linhas_tab.append(linha)
tabela = pd.DataFrame(linhas_tab)
tabela.columns = ["Método", "Acurácia", "Precisão", "Recall", "Especificidade", "Acur. balanceada", "F1",
                  "Baseline", "Acurácia − baseline", "Limiares por fold"]
display(tabela); tabela.to_csv(SAIDA/"tabela_final_image_level.csv", index=False)

agreg = []
for metodo, g in res.groupby("metodo", sort=False):
    TP, FP, FN, TN = (int(g[c].sum()) for c in ["TP", "FP", "FN", "TN"]); N = TP + FP + FN + TN
    lo, hi = wilson(TP + TN, N)
    agreg.append({"Método": metodo, "TP": TP, "FP": FP, "FN": FN, "TN": TN,
                  "Acurácia agregada": f"{(TP+TN)/N:.2%}", "IC 95% (Wilson)": f"[{lo:.1%}, {hi:.1%}]",
                  "Recall": f"{TP/(TP+FN):.2%}", "Especificidade": f"{TN/(TN+FP):.2%}",
                  "Baseline": f"{(TP+FN)/N:.2%}"})
agreg = pd.DataFrame(agreg); display(agreg); agreg.to_csv(SAIDA/"agregado_4folds.csv", index=False)

print("\nBox-Level (média ± desvio):")
display(pd.DataFrame({k_: [f"{box[c].mean():.2%} ± {box[c].std(ddof=1):.2%}"]
                      for k_, c in [("Precisão","precisao"),("Recall","recall"),("mAP@50","mAP50"),("mAP@50-95","mAP50_95")]}))

,Método,Acurácia,Precisão,Recall,Especificidade,Acur. balanceada,F1,Baseline,Acurácia − baseline,Limiares por fold
0,conf 0.50 (estudo anterior),62.87% ± 13.26%,95.95% ± 3.41%,57.48% ± 16.13%,88.78% ± 6.76%,73.13% ± 8.90%,71.05% ± 13.18%,83.51%,-20.64 p.p.,"0.500, 0.500, 0.500, 0.500"
1,conf 0.25 (padrão YOLO),82.89% ± 5.03%,93.31% ± 3.85%,85.70% ± 5.98%,67.81% ± 18.77%,76.75% ± 8.32%,89.22% ± 3.44%,83.51%,-0.62 p.p.,"0.250, 0.250, 0.250, 0.250"
2,limiar escolhido na validação (acurácia),87.08% ± 3.49%,88.44% ± 4.93%,97.63% ± 2.81%,33.24% ± 31.13%,65.43% ± 14.21%,92.69% ± 1.75%,83.51%,+3.57 p.p.,"0.100, 0.020, 0.050, 0.001"
3,limiar escolhido na validação (acur. balanceada),76.16% ± 12.78%,93.93% ± 3.70%,76.13% ± 14.78%,76.89% ± 9.17%,76.51% ± 9.70%,83.62% ± 10.13%,83.51%,-7.35 p.p.,"0.125, 0.550, 0.250, 0.400"


,Método,TP,FP,FN,TN,Acurácia agregada,IC 95% (Wilson),Recall,Especificidade,Baseline
0,conf 0.50 (estudo anterior),424,16,307,128,63.09%,"[59.8%, 66.2%]",58.00%,88.89%,83.54%
1,conf 0.25 (padrão YOLO),628,46,103,98,82.97%,"[80.3%, 85.3%]",85.91%,68.06%,83.54%
2,limiar escolhido na validação (acurácia),714,96,17,48,87.09%,"[84.7%, 89.1%]",97.67%,33.33%,83.54%
3,limiar escolhido na validação (acur. balanceada),556,34,175,110,76.11%,"[73.2%, 78.8%]",76.06%,76.39%,83.54%



Box-Level (média ± desvio):


,Precisão,Recall,mAP@50,mAP@50-95
0,56.86% ± 12.21%,45.30% ± 2.34%,44.98% ± 7.77%,18.57% ± 4.22%


### Capacidade de medir tutbarão e não-tubarão

In [8]:
from sklearn.metrics import roc_auc_score, average_precision_score
lin = []
for d in dados:
    y, s = np.array(d["test"]["y"]), np.array(d["test"]["score"])
    lin.append({"fold": d["fold"], "AUC-ROC": roc_auc_score(y, s),
                "AUC-PR (tubarão)": average_precision_score(y, s),
                "prevalência (referência do AUC-PR)": y.mean()})
auc = pd.DataFrame(lin); display(auc.round(3))
print(f"AUC-ROC média: {auc['AUC-ROC'].mean():.3f} ± {auc['AUC-ROC'].std(ddof=1):.3f}")
auc.to_csv(SAIDA/"auc_por_fold.csv", index=False)

,fold,AUC-ROC,AUC-PR (tubarão),prevalência (referência do AUC-PR)
0,0,0.915,0.979,0.832
1,1,0.885,0.974,0.863
2,2,0.898,0.974,0.826
3,3,0.699,0.901,0.819


AUC-ROC média: 0.849 ± 0.101


## 5. Comparar os experimentos (rode depois de concluir 2 ou mais)

A tabela traz, para cada configuração, a acurácia no **teste** e a acurácia na **validação interna**. **A configuração vencedora deve ser escolhida pela coluna de validação.** No TCC, apresente todas as configurações testadas (inclusive as que não venceram).

In [9]:
import glob
partes = [pd.read_csv(p) for p in sorted(glob.glob(str(BASE/"*"/"resultados_por_fold.csv")))]
todos = pd.concat(partes, ignore_index=True)

linhas = []
for (exp, metodo), g in todos.groupby(["experimento", "metodo"], sort=False):
    if metodo not in (M_ACC, M_BAL, "conf 0.25 (padrão YOLO)"):
        continue
    linhas.append({"Experimento": exp, "Método": metodo,
                   "Acurácia VALIDAÇÃO (critério de escolha)": f"{g.val_acuracia.mean():.2%}",
                   "Acurácia TESTE": f"{g.acuracia.mean():.2%} ± {g.acuracia.std(ddof=1):.2%}",
                   "Recall": f"{g.recall.mean():.2%}", "Especificidade": f"{g.especificidade.mean():.2%}",
                   "Acur. balanceada": f"{g.acuracia_balanceada.mean():.2%}",
                   "Baseline": f"{g.baseline.mean():.2%}"})
comp = pd.DataFrame(linhas)
display(comp)
comp.to_csv(BASE/"comparacao_experimentos.csv", index=False)

sel = (todos[todos.metodo == M_ACC].groupby("experimento").val_acuracia.mean().sort_values(ascending=False))
print("\nRanking pela acurácia na VALIDAÇÃO (regra de escolha):")
print(sel.map("{:.2%}".format).to_string())
print(f"\nConfiguração escolhida pela validação: {sel.index[0]}")

,Experimento,Método,Acurácia VALIDAÇÃO (critério de escolha),Acurácia TESTE,Recall,Especificidade,Acur. balanceada,Baseline
0,yolo11m_640,conf 0.25 (padrão YOLO),79.54%,82.89% ± 5.03%,85.70%,67.81%,76.75%,83.51%
1,yolo11m_640,limiar escolhido na validação (acurácia),87.58%,87.08% ± 3.49%,97.63%,33.24%,65.43%,83.51%
2,yolo11m_640,limiar escolhido na validação (acur. balanceada),77.71%,76.16% ± 12.78%,76.13%,76.89%,76.51%,83.51%
3,yolo11s_640,conf 0.25 (padrão YOLO),73.72%,79.26% ± 9.42%,79.85%,75.97%,77.91%,83.51%
4,yolo11s_640,limiar escolhido na validação (acurácia),89.63%,87.44% ± 5.08%,98.89%,31.15%,65.02%,83.51%
5,yolo11s_640,limiar escolhido na validação (acur. balanceada),82.18%,77.47% ± 12.75%,81.22%,61.72%,71.47%,83.51%
6,yolo11s_960,conf 0.25 (padrão YOLO),81.31%,82.47% ± 9.40%,85.70%,66.81%,76.25%,83.51%
7,yolo11s_960,limiar escolhido na validação (acurácia),88.01%,84.40% ± 2.87%,94.12%,33.83%,63.98%,83.51%
8,yolo11s_960,limiar escolhido na validação (acur. balanceada),79.23%,75.20% ± 6.07%,75.18%,76.27%,75.73%,83.51%



Ranking pela acurácia na VALIDAÇÃO (regra de escolha):
experimento
yolo11s_640    89.63%
yolo11s_960    88.01%
yolo11m_640    87.58%

Configuração escolhida pela validação: yolo11s_640


## Como reportar

- Descreva o protocolo: configurações testadas (`yolo11s_640`, `yolo11s_960`, `yolo11m_640`), escolha da configuração pela **validação interna**, folds idênticos entre experimentos.
- Apresente **todas** as configurações na mesma tabela, com acurácia, baseline, Recall e especificidade.
- A regra de decisão principal continua sendo a que você declarou antes (limiar escolhido na validação).